# Cost-matched rule baselines

Can a simple local rule achieve the same pollution level as Q-learning at a similar mitigation cost? This notebook compares the saved 20-seed Q-learning policies at spillover 0.2 with **all 15 monotone four-bin rules**. It uses the same observation available to each learned agent.

The rule choices and development seeds are fixed below. Development demand seeds select the rule nearest to each learned policy's *cost*. The original held-out seeds measure outcomes. Selection never uses pollution. This is an exploratory follow-up to results already seen on the held-out seeds, so it is not a new confirmatory test. Run from the `notebooks` directory.

In [ ]:
import json
from itertools import combinations_with_replacement

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from config import EnvironmentConfig, TrainingConfig
from evaluate import evaluate_policy, summarize_by_policy, summarize_by_training_seed, write_csv
from run_followup import greedy_actions, load_agents

In [ ]:
spillover = 0.2
development_seeds = tuple(range(200000, 200020))
evaluation_seeds = tuple(range(100000, 100020))
training_seeds = (11, 23, 37, 41, 53, 67, 71, 83, 97, 101, 107,
                  109, 127, 131, 137, 149, 157, 163, 173, 179)
reward_modes = ("local", "global", "mixed")
saved_models_directory = "../results/followup/spillover_02"
output_directory = "../results/notebook_runs/cost_matched_baselines_01"
save_results = False

environment_config = EnvironmentConfig()
environment_config.spillover = spillover
training_config = TrainingConfig()
rules = tuple(combinations_with_replacement(range(3), 4))
rule_names = {"rule_" + "".join(map(str, rule)): rule for rule in rules}

with open(f"{saved_models_directory}/manifest.json", encoding="utf-8") as file:
    saved_manifest = json.load(file)
if saved_manifest["environment"]["spillover"] != spillover:
    raise ValueError("The saved models use a different spillover.")
if tuple(saved_manifest["training"]["training_seeds"]) != training_seeds:
    raise ValueError("The saved models use different training seeds.")
if tuple(saved_manifest["training"]["evaluation_seeds"]) != evaluation_seeds:
    raise ValueError("The saved models use different evaluation seeds.")
if set(development_seeds) & set(evaluation_seeds):
    raise ValueError("Development and evaluation seeds must be separate.")

print(f"{len(rule_names)} rules; {len(training_seeds)} learned runs; "
      f"{len(development_seeds)} development and {len(evaluation_seeds)} held-out demand seeds.")

## Candidate rules

A rule is a four-digit action map for pollution bins 0–3. For example, `rule_0012` is the existing threshold baseline. Nondecreasing digits mean the region never chooses *less* mitigation in a higher pollution bin. The candidate list is fixed before comparing outcomes, and no rule sees neighbors' state.

In [ ]:
display(pd.DataFrame([
    {"policy": name, "bin_0": rule[0], "bin_1": rule[1],
     "bin_2": rule[2], "bin_3": rule[3]}
    for name, rule in rule_names.items()
]))

## Select cost matches using development demand seeds

The saved Q-tables are evaluated greedily. A single deterministic rule is evaluated once per demand seed, while each learned reward condition has 20 separately trained policies. We select by *mitigation cost only*; lower pollution cannot influence the match.

In [ ]:
development_rows = []

for name, rule in rule_names.items():
    policy = lambda observations, rule=rule: tuple(rule[observation[1]]
                                                     for observation in observations)
    for metrics in evaluate_policy(policy, development_seeds, environment_config):
        development_rows.append({"policy": name, "training_seed": 0, **metrics})

for mode in reward_modes:
    for seed in training_seeds:
        agents = load_agents(f"{saved_models_directory}/q_{mode}_{seed}.json",
                             seed, training_config)
        policy = lambda observations, agents=agents: greedy_actions(observations, agents)
        for metrics in evaluate_policy(policy, development_seeds, environment_config):
            development_rows.append({"policy": f"iql_{mode}", "training_seed": seed, **metrics})

development_runs = summarize_by_training_seed(development_rows)
development_summary = pd.DataFrame(summarize_by_policy(development_runs))
development_cost = development_summary.set_index("policy")["mitigation_cost_mean"]
matched_rules = {}
for mode in reward_modes:
    learner = f"iql_{mode}"
    matched_rules[learner] = min(
        rule_names,
        key=lambda name: abs(development_cost[name] - development_cost[learner]),
    )

selection = pd.DataFrame([
    {"learner": learner, "matched_rule": rule,
     "development_learner_cost": development_cost[learner],
     "development_rule_cost": development_cost[rule],
     "development_cost_gap": development_cost[learner] - development_cost[rule]}
    for learner, rule in matched_rules.items()
])
display(selection.round(2))

## Compare all rules on held-out demand seeds

Every rule sees the same 20 demand sequences as the previously saved Q-learning evaluation. The original learned policy summaries are read from disk; no retraining takes place. Deterministic rule outcomes can vary across demand sequences even though they have no training-seed variation.

In [ ]:
evaluation_rows = []
for name, rule in rule_names.items():
    policy = lambda observations, rule=rule: tuple(rule[observation[1]]
                                                     for observation in observations)
    for metrics in evaluate_policy(policy, evaluation_seeds, environment_config):
        evaluation_rows.append({"policy": name, "training_seed": 0, **metrics})

rule_runs = summarize_by_training_seed(evaluation_rows)
rule_summary = pd.DataFrame(summarize_by_policy(rule_runs))
learned_summary = pd.read_csv(f"{saved_models_directory}/policy_summary.csv")
learned_summary = learned_summary[learned_summary["policy"].isin(matched_rules)]
comparison = pd.concat([rule_summary, learned_summary], ignore_index=True)
comparison = comparison.sort_values("mitigation_cost_mean")

columns = ["policy", "mean_regional_pollution_mean", "mitigation_cost_mean",
           "threshold_exceedance_mean", "emitted_units_mean",
           "regional_pollution_spread_mean"]
display(comparison[columns].round(2))

In [ ]:
heldout = comparison.set_index("policy")
matched_comparison = selection.copy()
matched_comparison["heldout_learner_pollution"] = [
    heldout.loc[name, "mean_regional_pollution_mean"] for name in matched_comparison["learner"]
]
matched_comparison["heldout_rule_pollution"] = [
    heldout.loc[name, "mean_regional_pollution_mean"] for name in matched_comparison["matched_rule"]
]
matched_comparison["heldout_pollution_gap"] = (
    matched_comparison["heldout_learner_pollution"]
    - matched_comparison["heldout_rule_pollution"]
)
matched_comparison["heldout_cost_gap"] = [
    heldout.loc[learner, "mitigation_cost_mean"]
    - heldout.loc[rule, "mitigation_cost_mean"]
    for learner, rule in zip(matched_comparison["learner"],
                             matched_comparison["matched_rule"])
]
display(matched_comparison.round(2))

## Match expected cost with two-rule mixtures

A closer budget match can choose one of two fixed rules **at the start of an episode**. The choice is random, but the selected rule stays fixed for all 50 steps. Its expected outcome is the weighted average of the two complete rule outcomes. We find the nearest rule cost below and above each learned policy's development cost, then set the mixture weight to match that cost on development seeds. The held-out cost gap can still differ because demand sequences change.

In [ ]:
mixture_rows = []
for learner in matched_rules:
    target_cost = development_cost[learner]
    lower_rule = max((name for name in rule_names
                      if development_cost[name] <= target_cost),
                     key=lambda name: development_cost[name])
    upper_rule = min((name for name in rule_names
                      if development_cost[name] >= target_cost),
                     key=lambda name: development_cost[name])
    lower_cost = development_cost[lower_rule]
    upper_cost = development_cost[upper_rule]
    upper_fraction = ((target_cost - lower_cost) / (upper_cost - lower_cost)
                      if upper_cost > lower_cost else 0.0)

    lower_result = heldout.loc[lower_rule]
    upper_result = heldout.loc[upper_rule]
    mixture = {
        "learner": learner,
        "lower_rule": lower_rule,
        "upper_rule": upper_rule,
        "upper_fraction": upper_fraction,
        "development_target_cost": target_cost,
        "heldout_learner_cost": heldout.loc[learner, "mitigation_cost_mean"],
    }
    for metric in ("mean_regional_pollution", "mitigation_cost", "threshold_exceedance",
                   "emitted_units", "regional_pollution_spread"):
        column = f"{metric}_mean"
        mixture[f"heldout_mixture_{metric}"] = (
            (1 - upper_fraction) * lower_result[column]
            + upper_fraction * upper_result[column]
        )
    mixture["heldout_cost_gap"] = (
        mixture["heldout_learner_cost"] - mixture["heldout_mixture_mitigation_cost"]
    )
    mixture["heldout_pollution_gap"] = (
        heldout.loc[learner, "mean_regional_pollution_mean"]
        - mixture["heldout_mixture_mean_regional_pollution"]
    )
    mixture_rows.append(mixture)

mixture_comparison = pd.DataFrame(mixture_rows)
display(mixture_comparison[
    ["learner", "lower_rule", "upper_rule", "upper_fraction",
     "heldout_mixture_mitigation_cost", "heldout_cost_gap",
     "heldout_mixture_mean_regional_pollution", "heldout_pollution_gap"]
].round(2))

## Pollution versus mitigation cost

A point further left is cheaper; a point lower is less polluted. The pale rules are the complete candidate set, and hollow diamonds show the episode-level mixtures. Learned-policy error bars show one standard deviation across training seeds. A rule has no training-seed error bar; that does **not** mean demand sequences give identical outcomes. Compare the numeric held-out cost gaps above before interpreting pollution gaps.

In [ ]:
figure, axis = plt.subplots(figsize=(9, 6))
axis.scatter(rule_summary["mitigation_cost_mean"],
             rule_summary["mean_regional_pollution_mean"],
             color="lightgray", label="All monotone rules")

for learner, rule in matched_rules.items():
    rule_row = heldout.loc[rule]
    axis.scatter(rule_row["mitigation_cost_mean"],
                 rule_row["mean_regional_pollution_mean"],
                 s=90, marker="s", label=f"{rule} (match for {learner})")
    learner_row = heldout.loc[learner]
    axis.errorbar(learner_row["mitigation_cost_mean"],
                  learner_row["mean_regional_pollution_mean"],
                  xerr=learner_row["mitigation_cost_sd"],
                  yerr=learner_row["mean_regional_pollution_sd"],
                  fmt="o", capsize=4, label=learner)

for _, mixture in mixture_comparison.iterrows():
    axis.scatter(mixture["heldout_mixture_mitigation_cost"],
                 mixture["heldout_mixture_mean_regional_pollution"],
                 s=80, marker="D", facecolors="none", edgecolors="black",
                 label=f"mixture for {mixture['learner']}")

axis.set(xlabel="Mitigation cost per episode",
         ylabel="Mean regional pollution",
         title="Learned policies and simple rules at spillover 0.2")
axis.legend(fontsize=8)
axis.grid(alpha=0.2)
figure.tight_layout()
if save_results:
    figure.savefig(f"{output_directory}/pollution_vs_cost.svg")
display(figure)
plt.close(figure)

In [ ]:
learned_runs = pd.read_csv(f"{saved_models_directory}/run_summary.csv")
run_check = []
for _, mixture in mixture_comparison.iterrows():
    learner = mixture["learner"]
    learner_runs = learned_runs[learned_runs["policy"] == learner]
    pollution_gap = (learner_runs["mean_regional_pollution"]
                     - mixture["heldout_mixture_mean_regional_pollution"])
    run_check.append({"learner": learner,
                      "runs_with_lower_pollution": int((pollution_gap < 0).sum()),
                      "total_runs": len(learner_runs),
                      "pollution_gap_sd": pollution_gap.std()})
display(pd.DataFrame(run_check).round(2))

## Interpretation

On held-out demand, the learned policies spent 2.91–7.26 fewer cost units per episode than their matched rule mixtures, a small remaining difference in average cost. At these roughly matched budgets, local-, global-, and mixed-reward Q-learning had 0.11, 0.82, and 0.19 **more** mean pollution, respectively. The learned policy had lower pollution than its mixture in 8/20 local runs, 4/20 global runs, and 10/20 mixed runs. The global result is the clearest pattern; the local and mixed gaps are small and vary across training seeds.

Shared rewards changed agent choices and raised mitigation spending in the original comparison, but the learned policies did not beat these cost-matched rule mixtures on mean pollution in this synthetic setting. That weakens a claim that the global-reward result demonstrates a unique benefit from multi-agent learning. It does **not** show that every simple rule is better, or that reward design has no effect. We examined a limited set of rules, selected matches using development demand seeds, and matched costs on average rather than for each trained policy. Individual learned runs can spend very different amounts. This is an exploratory follow-up to previously viewed held-out results, so these differences should be presented as descriptive rather than a new confirmatory finding.

## Save the experiment record

The saved outputs can be viewed without changing files: `save_results = False` is the default. To save another run, prepare a new output folder, update `output_directory`, and set `save_results = True`. The manifest then opens in exclusive mode so an existing record is not silently overwritten. All paths stay relative to the `notebooks` directory.

In [ ]:
if save_results:
    manifest = {
        "question": "Cost-matched monotone rule baselines",
        "source": "notebooks/04_cost_matched_baselines.ipynb",
        "saved_models_directory": saved_models_directory,
        "environment": vars(environment_config),
        "training_seeds": list(training_seeds),
        "development_seeds": list(development_seeds),
        "evaluation_seeds": list(evaluation_seeds),
        "candidate_rules": {name: list(rule) for name, rule in rule_names.items()},
        "selected_matches": matched_rules,
        "mixtures": mixture_comparison[["learner", "lower_rule", "upper_rule",
                                        "upper_fraction"]].to_dict(orient="records"),
        "selection_metric": "Mean mitigation cost on development demand seeds",
        "status": "Exploratory follow-up",
    }
    with open(f"{output_directory}/manifest.json", "x", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
        file.write("\n")

    write_csv(f"{output_directory}/development_episodes.csv", development_rows)
    write_csv(f"{output_directory}/development_runs.csv", development_runs)
    development_summary.to_csv(f"{output_directory}/development_summary.csv", index=False)
    write_csv(f"{output_directory}/heldout_rule_episodes.csv", evaluation_rows)
    rule_summary.to_csv(f"{output_directory}/heldout_rule_summary.csv", index=False)
    matched_comparison.to_csv(f"{output_directory}/matched_comparison.csv", index=False)
    mixture_comparison.to_csv(f"{output_directory}/mixture_comparison.csv", index=False)
    print(f"Saved experiment tables and manifest to {output_directory}")
else:
    print("Analysis displayed; saved results remain unchanged.")